In [ ]:
import logging
import json
import numpy as np
import inspect
import numpy as np
import pandas as pd

import random
from typing import Dict, List, Optional, Tuple, Any

from pathlib import Path
from typing import Optional, Dict, Any

from utils.paths import get_paths
from utils.file_io import save_data
from utils.logging_setup import configure_logging, log_layer_paths
from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.synthetic_profiles import (
    load_rich_profile_csv,
    load_and_merge_rich_profiles,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)

from utils.synthetic_missingness import build_missingness_spec_from_truth_payload

from utils.synthetic_generator import SyntheticGenerator, EpisodeSpec

from utils.synthetic_postgres_writer import (
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    write_stream_batch,
)

from utils.synthetic_export import export_synthetic_batch_to_parquet
from utils.pipeline_config_loader import build_truth_config_block

from utils.postgres_util import get_engine_from_env, build_postgres_url, execute_sql, read_sql_dataframe
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe


In [ ]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

In [ ]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)
CONFIG = config_obj.data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})

PIPELINE_MODE = PIPELINE["execution_mode"]
DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = CONFIG["versions"]["truth"]

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])


TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)



set_wandb_dir_from_config(CONFIG)

print("DATASET_NAME:", DATASET_NAME)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

DATASET_NAME: pump
TRUTHS_PATH: /workspace/artifacts/truths
ARTIFACTS_ROOT: /workspace/artifacts


In [ ]:
# Logging Setup

# Create gold log path 
synthetic_log_path = paths.logs / "synthetic_data_generator.log"

# Initial Logger
configure_logging(
    "capstone",
    synthetic_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.synthetic")

# Log load and initiation
logger.info("Synethetic Data Generation starting")

# Log paths loads
log_layer_paths(paths, current_layer="synthetic", logger=logger)


2026-03-18 21:16:11,248 | INFO | capstone.synthetic | Synethetic Data Generation starting
2026-03-18 21:16:11,251 | INFO | capstone.synthetic | Project Root Path Loaded: /workspace
2026-03-18 21:16:11,253 | INFO | capstone.synthetic | Project Logging Path Loaded: /workspace/logs
2026-03-18 21:16:11,255 | INFO | capstone.synthetic | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-18 21:16:11,257 | INFO | capstone.synthetic | Project Notebooks Path Loaded: /workspace/notebooks
2026-03-18 21:16:11,259 | INFO | capstone.synthetic | Project Truths Path Loaded: /workspace/artifacts/truths
2026-03-18 21:16:11,261 | INFO | capstone.synthetic | Project Data Path Loaded: /workspace/data
2026-03-18 21:16:11,265 | INFO | capstone.synthetic | Previous Layer (Gold) Path Loaded: /workspace/data/gold


In [ ]:
def sample_int_range(rng: np.random.Generator, value_or_range, *, low_inclusive: bool = True) -> int:
    """
    Accepts either:
      - int
      - [low, high]
    Returns an int.
    """
    if isinstance(value_or_range, (int, np.integer)):
        return int(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = int(value_or_range[0])
        high = int(value_or_range[1])
        if low_inclusive:
            # numpy integers are low inclusive, high exclusive
            return int(rng.integers(low, high + 1))
        return int(rng.integers(low, high))

    raise TypeError(f"Expected int or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")


def sample_float_range(rng: np.random.Generator, value_or_range) -> float:
    """
    Accepts either:
      - float/int
      - [low, high]
    Returns a float.
    """
    if isinstance(value_or_range, (float, int, np.floating, np.integer)):
        return float(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = float(value_or_range[0])
        high = float(value_or_range[1])
        return float(rng.uniform(low, high))

    raise TypeError(f"Expected float or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")

In [ ]:
# Get Latest Truth Hash

def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> str:
    """
    Return the most recent truth_hash for a given layer + dataset from truth_index.jsonl.

    Assumes truth_index.jsonl is append-only and newer entries are later in the file.
    """
    if not truth_index_path.exists():
        raise FileNotFoundError(f"truth_index.jsonl not found: {truth_index_path}")

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record: Optional[Dict[str, Any]] = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            rec_layer = str(rec.get("layer_name", "")).strip().lower()
            rec_dataset = str(rec.get("dataset_name", "")).strip().lower()

            if rec_layer == layer_name_norm and rec_dataset == dataset_name_norm:
                latest_record = rec

    if latest_record is None:
        raise ValueError(
            f"No truth records found for layer='{layer_name}' dataset='{dataset_name}' in {truth_index_path}"
        )

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        raise ValueError("Latest record is missing a usable truth_hash.")

    return str(truth_hash).strip()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


def get_latest_silver_eda_truth_hash(
    *,
    truth_index_path: Path,
    dataset_name: str,
) -> str:
    """
    Convenience wrapper: latest Silver EDA truth hash for a dataset.
    """
    return get_latest_truth_hash(
        truth_index_path=truth_index_path,
        layer_name="silver_eda",
        dataset_name=dataset_name,
    )

In [ ]:
# Updated
# --- Notebook params ---
STAGE = SYN_CFG["layer_name"]
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# Parent truth hash from your latest Silver EDA run
SILVER_EDA_TRUTH_HASH = get_latest_silver_eda_truth_hash(truth_index_path=TRUTH_INDEX_PATH, dataset_name=DATASET_NAME,)
print("Latest SILVER_EDA_TRUTH_HASH:", SILVER_EDA_TRUTH_HASH)

# Faults
# Episode overrides (easy test knobs)
PRIMARY_SENSOR = None          # None => first sensor
PRIMARY_FAULT_TYPE = list(SYN_CFG["faults"]["allowed"])



# Episode Settings
NORMAL_BEFORE = list(SYN_CFG["episode"]["normal_before_range"])
BUILDUP = list(SYN_CFG["episode"]["buildup_range"])
FAILURE = list(SYN_CFG["episode"]["failure_range"])
RECOVERY = list(SYN_CFG["episode"]["recovery_range"])
NORMAL_AFTER = list(SYN_CFG["episode"]["normal_after_range"])
MAGNITUDE = list(SYN_CFG["episode"]["magnitude_range"])

SYNTH_PROCESS_RUN_ID = make_process_run_id(SYN_CFG["process_run_id_prefix"])

# Outputs
OUTPUT_MODE = SYN_CFG["output_mode"]

# Postgres settings
PG_SCHEMA = str(SYN_CFG["postgres"]["schema"])
TABLE_ARTIFACT_NAME = str(SYN_CFG["postgres"]["table_artifact_name"])

# medallion naming: synthetic_<dataset>_<artifact_name>
ARTIFACT_NAME = "stream"       

# Export
EXPORT_ENABLED = bool(SYN_CFG["export"]["enabled"])
EXPORT_DIRECTORY = str(SYN_CFG["export"]["export_dir_key"])

Latest SILVER_EDA_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188


In [ ]:
if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)


PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)

SILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

missingness_payload = (silver_preeda_truth.get("runtime_facts", {}) or {}).get("missingness_quarantine", {}) or {}
missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)


parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)

2026-03-18 21:16:11,448 | INFO | capstone.synthetic | W&B PARENT_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188
2026-03-18 21:16:11,450 | INFO | capstone.synthetic | W&B PIPELINE_MODE: batch


PARENT_TRUTH_HASH: df2f416c588a68ab07bf03afa75c23587ffb7609aa2eaca7edf0c5ed018d3188
PIPELINE_MODE: batch


In [ ]:
keys = SYN_CFG["silver_eda_artifact_keys"]

profile_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_normal"])
profile_abnormal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_abnormal"])
profile_recovery_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_recovery"])

corr_pairs_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["corr_pairs_normal"])
group_map_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["group_map_normal"])
fault_pairings_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["fault_pairings_normal"])

print(profile_normal_path)
print(profile_abnormal_path)
print(profile_recovery_path)
print(corr_pairs_normal_path)
print(group_map_normal_path)
print(fault_pairings_normal_path)

logger.info("silver_eda_artifact_keys: %s", keys)


2026-03-18 21:16:11,469 | INFO | capstone.synthetic | silver_eda_artifact_keys: {'profile_normal': 'feature_profile_normal_path', 'profile_abnormal': 'feature_profile_abnormal_path', 'profile_recovery': 'feature_profile_recovery_path', 'corr_pairs_normal': 'sensor_correlation_pairs_normal_path', 'group_map_normal': 'sensor_group_map_normal_path', 'fault_pairings_normal': 'sensor_fault_pairings_normal_path'}


/workspace/artifacts/silver_eda/pump/feature_profile_normal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_abnormal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_recovery.csv
/workspace/artifacts/silver_eda/pump/sensor_correlation_pairs_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_group_map_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_fault_pairings_normal.csv


In [ ]:
profile_dir = Path(profile_normal_path).parent

dropped_profile_normal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__normal.csv"
dropped_profile_abnormal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__abnormal.csv"
dropped_profile_recovery = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__recovery.csv"

dropped_profile_normal_path = str(dropped_profile_normal) if dropped_profile_normal.exists() else None
dropped_profile_abnormal_path = str(dropped_profile_abnormal) if dropped_profile_abnormal.exists() else None
dropped_profile_recovery_path = str(dropped_profile_recovery) if dropped_profile_recovery.exists() else None

normal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_normal_path),
    state_scope="normal",
    dropped_profile_csv_path=dropped_profile_normal_path,
)

abnormal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_abnormal_path),
    state_scope="abnormal",
    dropped_profile_csv_path=dropped_profile_abnormal_path,
)

recovery_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_recovery_path),
    state_scope="recovery",
    dropped_profile_csv_path=dropped_profile_recovery_path,
)

print("Dropped profile files found:",
      dropped_profile_normal.exists(),
      dropped_profile_abnormal.exists(),
      dropped_profile_recovery.exists())


corr_pairs_df = load_correlation_pairs_csv(corr_pairs_normal_path)
group_map_df = load_group_map_csv(group_map_normal_path)
fault_pairings_df = load_fault_pairings_csv(fault_pairings_normal_path)

generator = SyntheticGenerator(
    normal_profiles=normal_profiles,
    abnormal_profiles=abnormal_profiles,
    recovery_profiles=recovery_profiles,
    correlation_pairs_dataframe=corr_pairs_df,
    group_map_dataframe=group_map_df,
    fault_pairings_dataframe=fault_pairings_df,
    random_seed=int(SYN_CFG["random_seed"]),
)

print("Sensors:", len(generator.sensors))
print("First sensors:", generator.sensors[:10])

logger.info("Generator Run")
logger.info("Generator Sensors Count: %s", len(generator.sensors))
logger.info("Generator Sensors List: %s", generator.sensors)

Dropped profile files found: True True True
Sensors: 52
First sensors: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09']


2026-03-18 21:16:11,942 | INFO | capstone.synthetic | Generator Run
2026-03-18 21:16:11,944 | INFO | capstone.synthetic | Generator Sensors Count: 52
2026-03-18 21:16:11,946 | INFO | capstone.synthetic | Generator Sensors List: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51']


In [ ]:


print(inspect.signature(SyntheticGenerator.__init__))

logger.info("Generator Signature Inspection: %s", inspect.signature(SyntheticGenerator.__init__))

2026-03-18 21:16:11,964 | INFO | capstone.synthetic | Generator Signature Inspection: (self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None) -> 'None'


(self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None) -> 'None'


In [ ]:
# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 10_000   # only used if MODE="batch"
MAX_EPISODES = 10_000  # safety cap

# ---- RNG + config ----
rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))
ep_cfg = SYN_CFG["episode"]

allowed_faults = (SYN_CFG.get("faults", {}) or {}).get("allowed", [])
if not allowed_faults:
    raise ValueError("SYN_CFG.faults.allowed is empty or missing")

def pick_fault(rng: np.random.Generator) -> str:
    if PRIMARY_FAULT_TYPE is None or str(PRIMARY_FAULT_TYPE).strip() == "":
        return str(rng.choice(allowed_faults))
    if isinstance(PRIMARY_FAULT_TYPE, (list, tuple)):
        return str(rng.choice(list(PRIMARY_FAULT_TYPE)))
    return str(PRIMARY_FAULT_TYPE)

def pick_sensor(rng: np.random.Generator) -> str:
    if PRIMARY_SENSOR is None or str(PRIMARY_SENSOR).strip() == "":
        return str(rng.choice(generator.sensors))
    return str(PRIMARY_SENSOR)

def sample_episode_spec(rng: np.random.Generator) -> EpisodeSpec:
    magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
    if isinstance(magnitude, (list, tuple)):
        magnitude = float(rng.choice(list(magnitude)))

    return EpisodeSpec(
        primary_sensor=pick_sensor(rng),
        primary_fault_type=pick_fault(rng),
        magnitude=float(magnitude),
        normal_before=int(sample_int_range(rng, ep_cfg["normal_before_range"])),
        buildup=int(sample_int_range(rng, ep_cfg["buildup_range"])),
        failure=int(sample_int_range(rng, ep_cfg["failure_range"])),
        recovery=int(sample_int_range(rng, ep_cfg["recovery_range"])),
        normal_after=int(sample_int_range(rng, ep_cfg["normal_after_range"])),
    )

In [ ]:
# This cell is the ONLY place that creates/overwrites synthetic_df.

if MODE == "single":
    episode = sample_episode_spec(rng)
    synthetic_df = generator.generate_episode(episode)

    print("MODE=single")
    print("Episode:", episode)
    print("Generated rows:", len(synthetic_df))
    display(synthetic_df.head(10))
    display(synthetic_df["stream_state"].value_counts())

elif MODE == "batch":
    chunks = []
    row_count = 0
    episode_count = 0

    while row_count < TARGET_ROWS and episode_count < MAX_EPISODES:
        spec = sample_episode_spec(rng)
        df_ep = generator.generate_episode(spec).copy()

        # traceability / debugging
        df_ep["meta__episode_id"] = episode_count
        df_ep["meta__primary_sensor"] = spec.primary_sensor
        df_ep["meta__primary_fault_type"] = spec.primary_fault_type
        df_ep["meta__magnitude"] = float(spec.magnitude)

        chunks.append(df_ep)
        row_count += len(df_ep)
        episode_count += 1

    if not chunks:
        raise RuntimeError("Batch generation produced no episodes (unexpected).")

    synthetic_df = pd.concat(chunks, ignore_index=True)

    # hard trim
    if len(synthetic_df) > TARGET_ROWS:
        synthetic_df = synthetic_df.iloc[:TARGET_ROWS].copy()

    print("MODE=batch")
    print("Episodes used:", episode_count)
    print("Generated rows:", len(synthetic_df))
    display(synthetic_df["stream_state"].value_counts())

else:
    raise ValueError("MODE must be 'single' or 'batch'")

/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x
/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x
/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guid

MODE=batch
Episodes used: 9
Generated rows: 10000


/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x
/workspace/utils/synthetic_generator.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x


stream_state
normal      6163
recovery    2203
buildup     1614
abnormal      20
Name: count, dtype: int64

In [ ]:
def table_exists(engine, *, schema: str, table_name: str) -> bool:
    sql = """
    SELECT EXISTS(
      SELECT 1
      FROM information_schema.tables
      WHERE table_schema = :schema AND table_name = :table
    ) AS exists_flag
    """
    df = read_sql_dataframe(engine, sql, params={"schema": schema, "table": table_name})
    return bool(df.loc[0, "exists_flag"])


def drop_table(engine, *, schema: str, table_name: str) -> None:
    execute_sql(engine, f'DROP TABLE IF EXISTS "{schema}"."{table_name}" CASCADE')


def get_max_batch_id(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> int:
    fq = f'"{schema}"."{table_name}"'
    df = read_sql_dataframe(engine, f"SELECT COALESCE(MAX({batch_col}), 0) AS max_batch_id FROM {fq}")
    return int(df.loc[0, "max_batch_id"])


def canonicalize_existing_batch_ids(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> Dict[str, int]:
    """
    Renumber existing batch ids in-place to 1..N, preserving ascending order.
    Example: [4,5,6,7] -> [1,2,3,4]
    """
    fq = f'"{schema}"."{table_name}"'

    sql = f"""
    WITH b AS (
        SELECT DISTINCT {batch_col} AS old_batch_id
        FROM {fq}
        WHERE {batch_col} IS NOT NULL
    ),
    m AS (
        SELECT old_batch_id,
               DENSE_RANK() OVER (ORDER BY old_batch_id ASC) AS new_batch_id
        FROM b
    )
    UPDATE {fq} t
    SET {batch_col} = m.new_batch_id
    FROM m
    WHERE t.{batch_col} = m.old_batch_id
    """
    execute_sql(engine, sql)

    df = read_sql_dataframe(
        engine,
        f"""
        SELECT
          COUNT(DISTINCT {batch_col}) AS distinct_batches,
          COALESCE(MIN({batch_col}), 0) AS min_batch_id,
          COALESCE(MAX({batch_col}), 0) AS max_batch_id
        FROM {fq}
        """
    )
    return {
        "distinct_batches": int(df.loc[0, "distinct_batches"]),
        "min_batch_id": int(df.loc[0, "min_batch_id"]),
        "max_batch_id": int(df.loc[0, "max_batch_id"]),
    }


def choose_batch_id(
    engine,
    *,
    schema: str,
    table_name: str,
    write_mode: str,    # "reset" | "append"
    append_mode: str,   # "continue" | "renumber" (only used if write_mode="append")
    batch_col: str = "batch_id",
) -> int:
    """
    Implements your exact behavior:

    - write_mode="reset":
        drop table, recreate later via writer, next batch_id = 1

    - write_mode="append", append_mode="continue":
        do NOT renumber; next batch_id = MAX(batch_id)+1

    - write_mode="append", append_mode="renumber":
        renumber existing batches to 1..N (in-place), then next batch_id = N+1
        Example existing [4,5,6,7], new insert becomes 5 while table becomes [1,2,3,4,5]
    """
    write_mode = str(write_mode).strip().lower()
    append_mode = str(append_mode).strip().lower()

    if write_mode == "reset":
        drop_table(engine, schema=schema, table_name=table_name)
        return 1

    if write_mode != "append":
        raise ValueError("write_mode must be 'reset' or 'append'")

    exists = table_exists(engine, schema=schema, table_name=table_name)
    if not exists:
        return 1

    if append_mode == "continue":
        return get_max_batch_id(engine, schema=schema, table_name=table_name, batch_col=batch_col) + 1

    if append_mode == "renumber":
        summary = canonicalize_existing_batch_ids(engine, schema=schema, table_name=table_name, batch_col=batch_col)
        return int(summary["max_batch_id"]) + 1

    raise ValueError("append_mode must be 'continue' or 'renumber'")

In [ ]:
engine = get_engine_from_env()

PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = "reset"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")

batch_id = choose_batch_id(
    engine,
    schema=PG_SCHEMA,
    table_name=TABLE_NAME,
    write_mode=WRITE_MODE,
    append_mode=APPEND_MODE,
    batch_col="batch_id",
)

print("Chosen batch_id:", batch_id)

# cycles can stay sequence-based for now
ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")
cycle_start = reserve_cycle_range(
    engine,
    schema=PG_SCHEMA,
    sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id",
    n_rows=len(synthetic_df),
)

table_written = write_stream_batch(
    engine,
    synthetic_df,
    dataset_name=DATASET_NAME,
    schema=PG_SCHEMA,
    artifact_name=ARTIFACT_NAME,
    batch_id=batch_id,
    cycle_start=cycle_start,
)

print("Wrote:", table_written, "batch_id:", batch_id, "cycle_start:", cycle_start)

Chosen batch_id: 1
[synthetic] Added 56 new columns to capstone.synthetic_pump_stream
Wrote: synthetic_pump_stream batch_id: 1 cycle_start: 10351


In [ ]:
import os
{k: os.environ.get(k) for k in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD","POSTGRES_DB","POSTGRES_USER","POSTGRES_PASSWORD"]}

{'DB_HOST': 'dcook_capstone_postgres',
 'DB_PORT': '5432',
 'DB_NAME': 'dcook_capstone_postgres_db',
 'DB_USER': 'dcook_admin',
 'DB_PASSWORD': 'V9m!pQ4z@H2eS7wK',
 'POSTGRES_DB': None,
 'POSTGRES_USER': None,
 'POSTGRES_PASSWORD': None}

In [ ]:
import importlib.util
print("psycopg:", importlib.util.find_spec("psycopg"))
print("psycopg2:", importlib.util.find_spec("psycopg2"))

psycopg: None
psycopg2: ModuleSpec(name='psycopg2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x71bdc1f53340>, origin='/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2/__init__.py', submodule_search_locations=['/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2'])


In [ ]:
process_run_id = make_process_run_id(str(SYN_CFG.get("process_run_id_prefix", "synthetic")))

synthetic_truth = initialize_layer_truth(
    truth_version=str(TRUTH_VERSION),
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
    process_run_id=process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

LAYER_NAME = "synthetic"

resolved_config_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
resolved_config_dir.mkdir(parents=True, exist_ok=True)

resolved_config_path = resolved_config_dir / f"{DATASET_NAME}__{LAYER_NAME}__resolved_config.yaml"

# export_config_snapshot requires a destination path
export_config_snapshot(CONFIG, destination=resolved_config_path)

print("Resolved config written to:", resolved_config_path)


synthetic_truth = update_truth_section(
    synthetic_truth,
    "config_snapshot",
    {
        # store the path you just wrote
        "resolved_config_path": str(resolved_config_path),

        # optional: small inline config view (JSON-safe)
        "truth_config_block": build_truth_config_block(CONFIG),

        # keep your synthetic config subset if you want
        "synthetic_cfg": SYN_CFG,
    },
)

synthetic_truth = update_truth_section(
    synthetic_truth,
    "runtime_facts",
    {
        "primary_sensor": (
            getattr(episode, "primary_sensor", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0]) if "meta__primary_sensor" in synthetic_df.columns else None)
        ),
        "primary_fault_type": (
            getattr(episode, "primary_fault_type", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0]) if "meta__primary_fault_type" in synthetic_df.columns else None)
        ),
        "episode": (
            episode.__dict__
            if "episode" in globals()
            else {
                "mode": str(MODE) if "MODE" in globals() else None,
                "episodes_used": int(episode_count) if "episode_count" in globals() else None,
                "target_rows": int(TARGET_ROWS) if "TARGET_ROWS" in globals() else None,
                "meta_episode_id_min": int(synthetic_df["meta__episode_id"].min()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_episode_id_max": int(synthetic_df["meta__episode_id"].max()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_primary_sensor_mode": (
                    str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0])
                    if "meta__primary_sensor" in synthetic_df.columns and not synthetic_df["meta__primary_sensor"].dropna().empty
                    else None
                ),
                "meta_primary_fault_type_mode": (
                    str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0])
                    if "meta__primary_fault_type" in synthetic_df.columns and not synthetic_df["meta__primary_fault_type"].dropna().empty
                    else None
                ),
            }
        ),
        "normal_before_range": ep_cfg["normal_before_range"],
        "normal_before_selection": int(NORMAL_BEFORE) if "NORMAL_BEFORE" in globals() else None,
        "buildup_range": ep_cfg["buildup_range"],
        "buildup_selection": int(NORMAL_BEFORE) if "NORMAL_BEFORE" in globals() else None,
        "failure_range": ep_cfg["failure_range"],
        "failure_selection": int(FAILURE) if "NORMAL_BEFORE" in globals() else None,
        "recovery_range": ep_cfg["recovery_range"],
        "recovery_selection": int(RECOVERY) if "NORMAL_BEFORE" in globals() else None,
        "normal_after_range": ep_cfg["normal_after_range"],
        "normal_after_selection": int(NORMAL_AFTER) if "NORMAL_BEFORE" in globals() else None,
        "magnitude_range": ep_cfg["magnitude_range"],
        "magnitude_selection": float(MAGNITUDE) if "NORMAL_BEFORE" in globals() else None,
        "row_count": int(len(synthetic_df)),
        "parent_truth_hash": PARENT_TRUTH_HASH,
        "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    },
)

# Optionally save local parquet artifact too (useful for debugging)
synth_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
synth_dir.mkdir(parents=True, exist_ok=True)
suffix = "episode" if ("MODE" in globals() and str(MODE) == "single") else "batch"
out_path = synth_dir / f"{DATASET_NAME}__synthetic__{suffix}.parquet"
save_data(synthetic_df, synth_dir, out_path.name)

artifact_paths_payload = {
    "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    "profile_normal_path": str(profile_normal_path),
    "profile_abnormal_path": str(profile_abnormal_path),
    "profile_recovery_path": str(profile_recovery_path),
    "corr_pairs_normal_path": corr_pairs_normal_path,
    "group_map_normal_path": group_map_normal_path,
    "fault_pairings_normal_path": fault_pairings_normal_path,
    "synthetic_episode_path": str(out_path),
    "postgres_schema": PG_SCHEMA,
    "postgres_table": (
        table_written if "table_written" in globals()
        else (table_name if "table_name" in globals() else None)
    ),  
    "postgres_batch_id": int(batch_id),
    "postgres_cycle_start": int(cycle_start),
}

#export_path = Path(PATHS["data_raw_dir"] / "synthetic") 

if EXPORT_ENABLED:
    artifact_paths_payload["export_batch_parquet_path"] = str(out_path)

synthetic_truth = update_truth_section(synthetic_truth, "artifact_paths", artifact_paths_payload)

meta_columns = sorted(["meta__truth_hash", "meta__parent_truth_hash", "meta__pipeline_mode"])
feature_columns = sorted([c for c in synthetic_df.columns if not str(c).startswith("meta__")])

truth_record = build_truth_record(
    truth_base=synthetic_truth,
    row_count=int(len(synthetic_df)),
    column_count=int(synthetic_df.shape[1] + 3),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

synth_truth_hash = truth_record["truth_hash"]

# stamp lineage columns into dataframe (optional)
synthetic_df = stamp_truth_columns(
    synthetic_df,
    truth_hash=synth_truth_hash,
    parent_truth_hash=PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
)

append_truth_index(truth_record, truth_index_path=TRUTH_INDEX_PATH)

print("Synthetic truth hash:", synth_truth_hash)
print("Synthetic truth path:", truth_path)
print("Local episode parquet:", out_path)

Resolved config written to: /workspace/artifacts/synthetic/pump/pump__synthetic__resolved_config.yaml


NameError: name 'episode' is not defined

In [ ]:
from utils.synthetic_profiles import load_and_merge_rich_profiles
from utils.synthetic_generator import SyntheticGenerator, EpisodeSpec
from utils.synthetic_missingness import build_missingness_spec_from_truth_payload
from utils.synthetic_export import export_synthetic_batch_to_parquet

print("Imports OK")

In [ ]:
print("Episodes:", synthetic_df["meta__episode_id"].nunique() if "meta__episode_id" in synthetic_df else "NA")
print("Primary fault types:", synthetic_df["meta__primary_fault_type"].value_counts().head(10) if "meta__primary_fault_type" in synthetic_df else "NA")
print("Primary sensors:", synthetic_df["meta__primary_sensor"].value_counts().head(10) if "meta__primary_sensor" in synthetic_df else "NA")

In [ ]:
def verify_schema(
    df,
    *,
    expected_sensors: list[str],
    state_col: str = "stream_state",
    phase_col: str = "phase",
) -> dict:
    cols = set(df.columns)
    exp = set(expected_sensors)

    missing_cols = sorted(exp - cols)
    extra_sensor_cols = sorted([c for c in cols if c.startswith("sensor_") and c not in exp])

    out = {
        "row_count": int(len(df)),
        "missing_sensor_columns": missing_cols,
        "extra_sensor_columns": extra_sensor_cols,
        "state_values": sorted(df[state_col].dropna().astype(str).unique()) if state_col in df.columns else None,
        "phase_values": sorted(df[phase_col].dropna().astype(str).unique()) if phase_col in df.columns else None,
    }
    return out

In [ ]:
def verify_missingness_exact(
    df: pd.DataFrame,
    *,
    target_missing_pct: dict[str, float],   # sensor -> percent (0..100)
    sensors: list[str],
    tol_pct_points: float = 0.0,            # set 0 for exact; use small tol only for rounding
) -> pd.DataFrame:
    n = len(df)
    rows = []
    for s in sensors:
        if s not in df.columns:
            continue
        actual = float(df[s].isna().mean() * 100.0)
        target = float(target_missing_pct.get(s, 0.0))
        diff = actual - target
        ok = abs(diff) <= tol_pct_points
        rows.append({"sensor": s, "target_pct": target, "actual_pct": actual, "diff_pct_points": diff, "ok": ok})
    return pd.DataFrame(rows).sort_values("diff_pct_points", key=lambda c: c.abs(), ascending=False)

In [ ]:
def verify_profile_bounds(
    df: pd.DataFrame,
    *,
    profile_df: pd.DataFrame,     # rows: sensor,state_scope,p01,p99,mean,std,...
    sensors: list[str],
    state_col: str = "stream_state",
    state_values: list[str] = ["normal","abnormal","recovery"],
    bound_cols: tuple[str, str] = ("p01","p99"),
) -> pd.DataFrame:
    p01_col, p99_col = bound_cols
    prof = profile_df.copy()
    prof["sensor"] = prof["sensor"].astype(str)
    prof["state_scope"] = prof["state_scope"].astype(str)

    rows = []
    for st in state_values:
        df_st = df.loc[df[state_col].astype(str) == st]
        if df_st.empty:
            continue

        for s in sensors:
            if s not in df_st.columns:
                continue

            # profile row
            p = prof.loc[(prof["sensor"] == s) & (prof["state_scope"] == st)]
            if p.empty:
                continue
            p = p.iloc[0]

            series = pd.to_numeric(df_st[s], errors="coerce")
            nonnull = series.dropna()
            if nonnull.empty:
                rows.append({"state": st, "sensor": s, "nonnull_n": 0, "pct_outside_bounds": np.nan})
                continue

            lo = float(p[p01_col])
            hi = float(p[p99_col])

            outside = ((nonnull < lo) | (nonnull > hi)).mean() * 100.0
            rows.append({
                "state": st,
                "sensor": s,
                "nonnull_n": int(nonnull.shape[0]),
                "p01": lo,
                "p99": hi,
                "pct_outside_bounds": float(outside),
                "mean_actual": float(nonnull.mean()),
                "std_actual": float(nonnull.std(ddof=1)) if nonnull.shape[0] > 1 else 0.0,
                "mean_profile": float(p["mean"]) if "mean" in p else np.nan,
                "std_profile": float(p["std"]) if "std" in p else np.nan,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["pct_outside_bounds","nonnull_n"], ascending=[False, True])

In [ ]:
def verify_top_correlations(
    df: pd.DataFrame,
    *,
    corr_pairs_df: pd.DataFrame,      # sensor_a,sensor_b,abs_pearson_corr,...
    sensors: list[str],
    state_col: str = "stream_state",
    state_value: str = "normal",
    top_k: int = 25,
) -> pd.DataFrame:
    df_n = df.loc[df[state_col].astype(str) == state_value, sensors].copy()
    df_n = df_n.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    pear = df_n.corr(method="pearson")

    pairs = corr_pairs_df.head(top_k).copy()
    rows = []
    for _, r in pairs.iterrows():
        a, b = str(r["sensor_a"]), str(r["sensor_b"])
        if a in pear.columns and b in pear.columns:
            actual = float(pear.loc[a, b])
            rows.append({
                "sensor_a": a,
                "sensor_b": b,
                "expected_abs": float(r.get("abs_pearson_corr", np.nan)),
                "actual_abs": float(abs(actual)),
                "actual_signed": actual,
            })
    return pd.DataFrame(rows).sort_values("expected_abs", ascending=False)